# LLM Benchmark Analysis & Visualization

**Project:** Selection and Benchmarking of a Large Language Model for Mettle's Socratic and Adaptive Chatbot
**Author:** Kshitish Kiran Madbhavi

This notebook processes the raw data from the `llm-harness` and the human evaluation pipeline to generate the final tables and figures for the research paper as outlined in Section 11.

## 1. Setup and Imports

First, we import the necessary Python libraries for data manipulation (`pandas`), numerical operations (`numpy`), and visualization (`matplotlib`, `seaborn`, `plotly`).

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

# Set a consistent style for the plots
sns.set_theme(style="whitegrid")

## 2. Load and Prepare Data

Here, we will load the two primary data sources:
1.  **`benchmark_results.csv`**: The raw output from the `main.py` harness, containing performance metrics like latency and token counts.
2.  **`human_evaluation.csv`**: The collated scores from the Google Form, containing the human ratings for pedagogical quality and contextual fidelity.

**Action Required:** Replace the placeholder dataframes below with your actual data by loading your CSV files.

In [ ]:
# --- ACTION REQUIRED: Load your actual data here ---
# Example: 
# harness_df = pd.read_csv('../results/raw_output/your_harness_output.csv')
# human_df = pd.read_csv('../results/raw_output/your_human_scores.csv')

# For now, we'll create placeholder data that mirrors your paper's structure.
models = ['GPT-4o mini', 'Claude 3 Sonnet', 'Gemini 1.5 Flash', 'Command R', 'Llama 3 8B']

harness_data = {
    'response_id': range(250),
    'model': np.repeat(models, 50),
    'latency_ms': np.random.lognormal(mean=6.5, sigma=0.8, size=250),
    'cost_usd': np.random.uniform(0.0001, 0.001, size=250) * np.repeat([1.2, 1.0, 0.3, 0.8, 0.25], 50)
}
harness_df = pd.DataFrame(harness_data)

human_data = {
    'response_id': range(250),
    'pedagogical_quality': np.random.randint(1, 6, size=250) * np.repeat([1.4, 1.3, 1.0, 1.1, 0.9], 50),
    'contextual_fidelity': np.random.randint(1, 6, size=250) * np.repeat([1.3, 1.4, 1.1, 1.2, 0.8], 50)
}
human_df = pd.DataFrame(human_data)
human_df['pedagogical_quality'] = human_df['pedagogical_quality'].clip(1, 5)
human_df['contextual_fidelity'] = human_df['contextual_fidelity'].clip(1, 5)

# Merge the two dataframes
df = pd.merge(harness_df, human_df, on='response_id')

print("Sample of merged data:")
df.head()

## 3. Aggregate Data and Normalize Scores

Now we'll process the merged data to get the final mean scores for each model, which will be used to fill **Table 5** in the paper. This includes normalizing cost and latency to a 1-5 scale for the radar chart.

In [ ]:
# Aggregate data by model
agg_df = df.groupby('model').agg(
    pedagogical_quality=('pedagogical_quality', 'mean'),
    contextual_fidelity=('contextual_fidelity', 'mean'),
    latency_ms=('latency_ms', 'median'), # Median is more robust to outliers for latency
    cost_per_interaction=('cost_usd', 'mean')
).reset_index()

# --- Normalize scores to a 1-5 scale ---
# For cost and latency, lower is better, so we invert the scale.
def normalize(series, invert=False):
    min_val, max_val = series.min(), series.max()
    if invert:
        # Invert so that lowest value gets highest score (5)
        normalized = 5 - 4 * (series - min_val) / (max_val - min_val)
    else:
        # Standard normalization where highest value gets highest score (5)
        normalized = 1 + 4 * (series - min_val) / (max_val - min_val)
    return normalized

agg_df['cost_score'] = normalize(agg_df['cost_per_interaction'], invert=True)
agg_df['latency_score'] = normalize(agg_df['latency_ms'], invert=True)

# For the scatter plot, we need cost per 1k interactions
agg_df['cost_per_1k_interactions'] = agg_df['cost_per_interaction'] * 1000

print("Aggregated and Normalized Scores (for Table 5 and Radar Chart):")
agg_df

## 4. Generate Visualizations (Section 11.2)

This section generates the three main figures required for the paper.

### Figure 2: Radar Chart

A radar chart to provide a holistic comparison of the models across key evaluation criteria. The ideal model would have a large, symmetrical shape.

In [ ]:
categories = ['Pedagogical Quality', 'Contextual Fidelity', 'Cost', 'Latency']
fig = go.Figure()

for i, row in agg_df.iterrows():
    model_name = row['model']
    values = [
        row['pedagogical_quality'], 
        row['contextual_fidelity'], 
        row['cost_score'], 
        row['latency_score']
    ]
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=categories,
        fill='toself',
        name=model_name
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[1, 5]
        )
    ),
    title='Figure 2: Radar Chart of Model Performance',
    showlegend=True
)

fig.show()

### Figure 3: Scatterplot (Cost vs. Quality)

This plot visualizes the critical cost-benefit trade-off. The most desirable models are in the top-left quadrant (high quality, low cost).

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=agg_df,
    x='cost_per_1k_interactions',
    y='pedagogical_quality',
    hue='model',
    s=200, # Set marker size
    alpha=0.8
)

# Add labels to each point
for i, row in agg_df.iterrows():
    plt.text(row['cost_per_1k_interactions'] + 0.01, row['pedagogical_quality'], row['model'])

plt.title('Figure 3: Trade-off between Pedagogical Quality and Cost')
plt.xlabel('Cost per 1,000 Interactions (USD)')
plt.ylabel('Mean Pedagogical Quality Score (1-5)')
plt.legend(title='Models', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Figure 4: CDF Plot of API Latency

A Cumulative Distribution Function (CDF) plot shows the percentage of requests that completed within a given time. A curve that is steeper and further to the left indicates a faster and more consistent user experience.

In [ ]:
plt.figure(figsize=(12, 7))
sns.ecdfplot(data=df, x='latency_ms', hue='model')

plt.title('Figure 4: Cumulative Distribution of API Latency')
plt.xlabel('Latency (ms)')
plt.ylabel('Proportion of Requests')
plt.xlim(0, df['latency_ms'].quantile(0.99)) # Zoom in on the 99th percentile for readability
plt.grid(True)
plt.show()